import pandas as pd

In [30]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd() / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"

input_path = DATA_DIR / "accuweather_data.json"
df_raw = pd.read_json(input_path)

print(f"Registros carregados: {len(df_raw)}")
df_raw.head()

Registros carregados: 2


,Version,Key,Type,Rank,LocalizedName,EnglishName,PrimaryPostalCode,Region,Country,AdministrativeArea,TimeZone,GeoPosition,IsAlias,SupplementalAdminAreas,DataSets
0,1,4651,City,20,Luanda,Luanda,,"{'ID': 'AFR', 'LocalizedName': 'Africa', 'Engl...","{'ID': 'AO', 'LocalizedName': 'Angola', 'Engli...","{'ID': 'LUA', 'LocalizedName': 'Luanda', 'Engl...","{'Code': 'WAT', 'Name': 'Africa/Luanda', 'GmtO...","{'Latitude': -8.812, 'Longitude': 13.235, 'Ele...",False,[],"[AirQualityCurrentConditions, AirQualityForeca..."
1,1,6777,City,105,Luanda,Luanda,,"{'ID': 'AFR', 'LocalizedName': 'Africa', 'Engl...","{'ID': 'AO', 'LocalizedName': 'Angola', 'Engli...","{'ID': 'UIG', 'LocalizedName': 'Uige', 'Englis...","{'Code': 'WAT', 'Name': 'Africa/Luanda', 'GmtO...","{'Latitude': -7.129, 'Longitude': 14.801, 'Ele...",False,[],"[AirQualityCurrentConditions, AirQualityForeca..."


In [31]:
df = pd.json_normalize(df_raw.to_dict(orient="records"), sep="_")

print(f"Colunas após normalização: {len(df.columns)}")
df.head()

Colunas após normalização: 36


,Version,Key,Type,Rank,LocalizedName,EnglishName,PrimaryPostalCode,IsAlias,SupplementalAdminAreas,DataSets,...,TimeZone_IsDaylightSaving,TimeZone_NextOffsetChange,GeoPosition_Latitude,GeoPosition_Longitude,GeoPosition_Elevation_Metric_Value,GeoPosition_Elevation_Metric_Unit,GeoPosition_Elevation_Metric_UnitType,GeoPosition_Elevation_Imperial_Value,GeoPosition_Elevation_Imperial_Unit,GeoPosition_Elevation_Imperial_UnitType
0,1,4651,City,20,Luanda,Luanda,,False,[],"[AirQualityCurrentConditions, AirQualityForeca...",...,False,None,-8.812,13.235,1.0,m,5,3.0,ft,0
1,1,6777,City,105,Luanda,Luanda,,False,[],"[AirQualityCurrentConditions, AirQualityForeca...",...,False,None,-7.129,14.801,601.0,m,5,1971.0,ft,0


In [32]:
selected_columns = [
    "Key",
    "LocalizedName",
    "EnglishName",
    "Rank",
    "Region_ID",
    "Region_EnglishName",
    "Country_ID",
    "Country_EnglishName",
    "AdministrativeArea_ID",
    "AdministrativeArea_EnglishName",
    "TimeZone_Name",
    "TimeZone_GmtOffset",
    "GeoPosition_Latitude",
    "GeoPosition_Longitude",
]

locations = df.loc[:, selected_columns].copy()
locations = locations.rename(
    columns={
        "Key": "location_key",
        "LocalizedName": "name_localized",
        "EnglishName": "name_english",
        "Rank": "rank",
        "Region_ID": "region_id",
        "Region_EnglishName": "region",
        "Country_ID": "country_id",
        "Country_EnglishName": "country",
        "AdministrativeArea_ID": "administrative_area_id",
        "AdministrativeArea_EnglishName": "administrative_area",
        "TimeZone_Name": "timezone",
        "TimeZone_GmtOffset": "utc_offset",
        "GeoPosition_Latitude": "latitude",
        "GeoPosition_Longitude": "longitude",
    }
)

locations[["latitude", "longitude", "utc_offset"]] = locations[
    ["latitude", "longitude", "utc_offset"]
].apply(pd.to_numeric, errors="coerce")
locations = locations.sort_values(["rank", "name_english"]).reset_index(drop=True)

output_csv = DATA_DIR / "locations_transformed.csv"
output_json = DATA_DIR / "locations_transformed.json"
locations.to_csv(output_csv, index=False)
locations.to_json(output_json, orient="records", force_ascii=False, indent=4)

print(f"Tabela transformada: {locations.shape[0]} linhas x {locations.shape[1]} colunas")
print(f"CSV: {output_csv}")
print(f"JSON: {output_json}")
locations

Tabela transformada: 2 linhas x 14 colunas
CSV: /mnt/c/users/zh-ferreira/Documents/DevLab/Python/accuweather-project/data/locations_transformed.csv
JSON: /mnt/c/users/zh-ferreira/Documents/DevLab/Python/accuweather-project/data/locations_transformed.json


,location_key,name_localized,name_english,rank,region_id,region,country_id,country,administrative_area_id,administrative_area,timezone,utc_offset,latitude,longitude
0,4651,Luanda,Luanda,20,AFR,Africa,AO,Angola,LUA,Luanda,Africa/Luanda,1.0,-8.812,13.235
1,6777,Luanda,Luanda,105,AFR,Africa,AO,Angola,UIG,Uige,Africa/Luanda,1.0,-7.129,14.801
